# 二次融合优化

`07_submission_xgboost_tuned_blend.csv` 的公榜分数为 `0.12516`，是当前最好成绩。本 notebook 不重新训练模型，只对已有强提交做预测层二次融合，尝试低风险微调。

由于 Kaggle 评价指标是 RMSLE，这里优先使用 `log1p` 空间融合。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SUBMISSION_DIR = ROOT / "submissions"

submission_files = {
    "tuned": SUBMISSION_DIR / "07_submission_xgboost_tuned_blend.csv",
    "weighted": SUBMISSION_DIR / "06_submission_weighted_blend.csv",
    "boosting": SUBMISSION_DIR / "05_submission_boosting_blend.csv",
    "improve": SUBMISSION_DIR / "03_submission_improve_from_baseline.csv",
    "advanced_linear": SUBMISSION_DIR / "04_submission_advanced_linear.csv",
}

predictions = None
for name, path in submission_files.items():
    submission = pd.read_csv(path).rename(columns={"SalePrice": name})
    predictions = submission if predictions is None else predictions.merge(submission, on="Id")

predictions.head()

,Id,tuned,weighted,boosting,improve,advanced_linear
0,1461,120984.059166,120500.729881,120722.803033,119536.317378,119982.559191
1,1462,153899.071413,154619.764962,154951.123889,154471.242875,153846.591528
2,1463,183783.485114,183403.994934,183777.315814,184430.801899,182532.912879
3,1464,196585.732435,197433.733614,196259.888837,199102.240023,200172.715175
4,1465,193484.108709,191987.176148,190432.693792,195387.835972,195614.309458


## 1. 提交文件相关性

`weighted` 与 `tuned` 几乎完全一致；`boosting` 稍微更有差异，而且最高预测更低，因此更适合少量混入当前最好提交。

In [2]:
score_reference = pd.DataFrame(
    [
        {"提交": "tuned", "公榜分数": 0.12516},
        {"提交": "weighted", "公榜分数": 0.12549},
        {"提交": "boosting", "公榜分数": 0.12585},
        {"提交": "improve", "公榜分数": 0.12629},
        {"提交": "advanced_linear", "公榜分数": 0.12729},
    ]
)
score_reference

,提交,公榜分数
0,tuned,0.12516
1,weighted,0.12549
2,boosting,0.12585
3,improve,0.12629
4,advanced_linear,0.12729


In [3]:
prediction_columns = list(submission_files)
np.log1p(predictions[prediction_columns]).corr().round(5)

,tuned,weighted,boosting,improve,advanced_linear
tuned,1.00000,0.99982,0.99949,0.99704,0.99845
weighted,0.99982,1.00000,0.99972,0.99712,0.99851
boosting,0.99949,0.99972,1.00000,0.99563,0.99695
improve,0.99704,0.99712,0.99563,1.00000,0.99844
advanced_linear,0.99845,0.99851,0.99695,0.99844,1.00000


In [4]:
diff_rows = []
for column in prediction_columns:
    if column == "tuned":
        continue
    diff = predictions[column] - predictions["tuned"]
    abs_pct = diff.abs() / predictions["tuned"] * 100
    diff_rows.append(
        {
            "提交": column,
            "平均差异": diff.mean(),
            "中位绝对百分比差异": abs_pct.median(),
            "95%绝对百分比差异": abs_pct.quantile(0.95),
            "最大绝对百分比差异": abs_pct.max(),
        }
    )

pd.DataFrame(diff_rows).round(4)

,提交,平均差异,中位绝对百分比差异,95%绝对百分比差异,最大绝对百分比差异
0,weighted,-19.8345,0.4275,1.5212,3.8612
1,boosting,-135.7198,0.7338,2.5956,5.8234
2,improve,378.7326,1.8190,6.2090,16.9619
3,advanced_linear,250.5675,1.4109,4.4039,16.5622


## 2. 构造候选二次融合

这里不做大范围搜索，只构造几种保守候选。最终选择 `85% tuned + 15% boosting` 的 log 空间融合，因为它以当前最好提交为主，同时引入一点更保守的 boosting 形状。

In [5]:
def log_blend(weighted_columns: dict[str, float]) -> pd.Series:
    blended = np.zeros(len(predictions))
    for column, weight in weighted_columns.items():
        blended += weight * np.log1p(predictions[column])
    return pd.Series(np.expm1(blended), index=predictions.index)


def linear_blend(weighted_columns: dict[str, float]) -> pd.Series:
    blended = np.zeros(len(predictions))
    for column, weight in weighted_columns.items():
        blended += weight * predictions[column]
    return pd.Series(blended, index=predictions.index)


candidate_specs = {
    "log_tuned_90_boosting_10": (log_blend, {"tuned": 0.90, "boosting": 0.10}),
    "log_tuned_85_boosting_15": (log_blend, {"tuned": 0.85, "boosting": 0.15}),
    "log_tuned_80_boosting_20": (log_blend, {"tuned": 0.80, "boosting": 0.20}),
    "log_tuned_80_weighted_20": (log_blend, {"tuned": 0.80, "weighted": 0.20}),
    "log_tuned_70_weighted_15_boosting_15": (
        log_blend,
        {"tuned": 0.70, "weighted": 0.15, "boosting": 0.15},
    ),
    "linear_tuned_85_boosting_15": (linear_blend, {"tuned": 0.85, "boosting": 0.15}),
}

candidate_predictions = {}
for name, (blend_function, weights) in candidate_specs.items():
    candidate_predictions[name] = blend_function(weights)

candidate_summary_rows = []
for name, values in candidate_predictions.items():
    abs_pct_diff = (values - predictions["tuned"]).abs() / predictions["tuned"] * 100
    candidate_summary_rows.append(
        {
            "候选": name,
            "最小值": values.min(),
            "中位数": values.median(),
            "最大值": values.max(),
            "相对 tuned 的中位差异%": abs_pct_diff.median(),
            "相对 tuned 的95%差异%": abs_pct_diff.quantile(0.95),
        }
    )

candidate_summary = pd.DataFrame(candidate_summary_rows).sort_values("相对 tuned 的95%差异%")
candidate_summary.round(4)

,候选,最小值,中位数,最大值,相对 tuned 的中位差异%,相对 tuned 的95%差异%
0,log_tuned_90_boosting_10,42262.6113,157673.8511,857803.6574,0.0732,0.2611
3,log_tuned_80_weighted_20,42305.6871,157679.4387,865133.3368,0.0854,0.3044
5,linear_tuned_85_boosting_15,42200.2951,157639.2384,856309.8518,0.1101,0.3893
1,log_tuned_85_boosting_15,42197.7770,157639.0427,856237.5580,0.1098,0.3914
2,log_tuned_80_boosting_20,42133.0422,157604.2419,854674.3180,0.1464,0.5216
4,log_tuned_70_weighted_15_boosting_15,42132.8911,157538.1944,859360.1474,0.1632,0.5759


## 3. 生成提交文件

选择 `log_tuned_85_boosting_15` 作为主提交。它不会明显偏离当前最好提交，但会轻微吸收 `boosting` 版本更保守的高价区预测。

In [6]:
selected_candidate = "log_tuned_85_boosting_15"
final_prediction = candidate_predictions[selected_candidate]

submission = pd.DataFrame({"Id": predictions["Id"], "SalePrice": final_prediction})
submission_path = SUBMISSION_DIR / "08_submission_second_level_blend.csv"
submission.to_csv(submission_path, index=False)

selected_candidate, submission_path, submission.head(), submission["SalePrice"].describe()

('log_tuned_85_boosting_15',
 WindowsPath('C:/Users/84740/house-prices-advanced-regression/submissions/08_submission_second_level_blend.csv'),
      Id      SalePrice
 0  1461  120944.834733
 1  1462  154056.422732
 2  1463  183782.559706
 3  1464  196536.821430
 4  1465  193023.298439,
 count      1459.000000
 mean     178709.573556
 std       78765.055301
 min       42197.776991
 25%      126388.998052
 50%      157639.042679
 75%      209690.213445
 max      856237.558046
 Name: SalePrice, dtype: float64)

## 4. 提交命令

```powershell
kaggle competitions submit -c house-prices-advanced-regression-techniques -f submissions/08_submission_second_level_blend.csv -m "second level tuned boosting log blend"
```